# M9.4 · Türevler ve keskinleştirme

Yüksek geçiren filtreler: birinci ve ikinci türevin kenar, rampa, ince çizgi ve nokta üzerindeki davranışını 1-boyutlu bir profilde elle hesaplıyor; saf Python'da Laplasyen ve Sobel'i OpenCV'nin kenar kuralıyla yazıp birebir karşılaştırıyor, NumPy'da Laplasyen keskinleştirme, unsharp masking, highboost ve gradyan büyüklüğünü kuruyor, PyTorch'ta türevlenebilir bir unsharp mask'in $\sigma$ ve kazancını hedef çıktıdan gradient descent ile geri buluyor ve en sonda bir görüntü yığını için gradyan alanı ve yönelim histogramlarını tek geçişte hesaplıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-4-sharpening.html](https://m-gonul.github.io/pytorch-lab/m9-4-sharpening.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_4_sharpening.py
CHECKS = {
 "b": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"cv2/numpy kullanılmıyor\", lambda: no_imports(src, {\"cv2\", \"numpy\", \"skimage\", \"scipy\"}))\n    f = [5, 5, 4, 3, 2, 1, 0, 0, 0, 6, 0, 0]\n\n    def d():\n        ensure(list(need(ns, \"diff1\")(f)) == [0, -1, -1, -1, -1, -1, 0, 0, 6, -6, 0], \"diff1\")\n        ensure(list(need(ns, \"diff2\")(f)) == [-1, 0, 0, 0, 0, 1, 0, 6, -12, 6], \"diff2\")\n    R.test(\"diff1/diff2 (dersteki profil)\", d)\n    crop = data.camera()[300:340, 50:96].astype(np.float32)\n\n    def lap():\n        got = np.array(need(ns, \"laplacian\")(crop.tolist()), np.float32)\n        close(got, cv2.Laplacian(crop, cv2.CV_32F, ksize=1), atol=1e-3)\n    R.test(\"laplacian = cv2.Laplacian(ksize=1), REFLECT_101\", lap)\n\n    def sob():\n        gx, gy = need(ns, \"sobel\")(crop.tolist())\n        close(np.array(gx, np.float32), cv2.Sobel(crop, cv2.CV_32F, 1, 0, ksize=3), atol=1e-3, what=\"gx\")\n        close(np.array(gy, np.float32), cv2.Sobel(crop, cv2.CV_32F, 0, 1, ksize=3), atol=1e-3, what=\"gy\")\n    R.test(\"sobel = cv2.Sobel(ksize=3)\", sob)\n",
 "c": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    img = data.moon()\n    f = img.astype(np.float32)\n\n    def sl():\n        for kind, K in ((4, np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float32)), (8, np.array([[1, 1, 1], [1, -8, 1], [1, 1, 1]], np.float32))):\n            want = np.clip(np.rint(f - cv2.filter2D(f, -1, K)), 0, 255).astype(np.uint8)\n            got = need(ns, \"sharpen_laplacian\")(img, kind)\n            ensure(got.dtype == np.uint8 and np.abs(got.astype(int) - want.astype(int)).max() <= 1, f\"kind = {kind}\")\n    R.test(\"sharpen_laplacian (4 ve 8 komşu)\", sl)\n\n    def us():\n        for k in (1.0, 2.5):\n            blur = cv2.GaussianBlur(f, (5, 5), 0)\n            want = np.clip(np.rint(f + k * (f - blur)), 0, 255).astype(np.uint8)\n            got = need(ns, \"unsharp\")(img, k)\n            ensure(np.abs(got.astype(int) - want.astype(int)).max() <= 1, f\"k = {k}\")\n    R.test(\"unsharp ve highboost\", us)\n\n    def gm():\n        gx = cv2.Sobel(f, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(f, cv2.CV_32F, 0, 1, ksize=3)\n        close(need(ns, \"gradient_magnitude\")(img, \"l2\"), cv2.magnitude(gx, gy), atol=1e-2, what=\"L2\")\n        close(ns[\"gradient_magnitude\"](img, \"l1\"), np.abs(gx) + np.abs(gy), atol=1e-2, what=\"L1\")\n    R.test(\"gradient_magnitude = cv2.magnitude(Sobel)\", gm)\n",
 "d": "import torch\nimport torch.nn.functional as F\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    x = torch.tensor(data.camera(), dtype=torch.float32)[::4, ::4] / 255\n\n    def gk():\n        s = torch.tensor(1.5, requires_grad=True)\n        g = need(ns, \"gaussian_kernel1d\")(s, 9)\n        close(g.sum(), 1.0, atol=1e-6)\n        ensure(g.argmax().item() == 4, \"tepe ortada\")\n        g[4].backward()\n        ensure(s.grad is not None and s.grad < 0, \"σ büyüdükçe merkez ağırlık azalır: dg/dσ < 0\")\n    R.test(\"gaussian_kernel1d türevlenebilir\", gk)\n\n    def fit():\n        m = need(ns, \"UnsharpMask\", \"sınıf\")()\n        with torch.no_grad():\n            m.log_sigma.fill_(torch.tensor(1.8).log()); m.amount.fill_(1.6)\n            target = m(x)\n        torch.manual_seed(0)\n        s, a = need(ns, \"fit_unsharp\")(x, target)\n        ensure(abs(s - 1.8) < 0.05 and abs(a - 1.6) < 0.05, f\"bulunan σ = {s:.3f}, kazanç = {a:.3f}; gerçek 1.8, 1.6\")\n        return f\"σ = {s:.3f} (1.8), kazanç = {a:.3f} (1.6)\"\n    R.test(\"fit_unsharp parametreleri geri buluyor\", fit)\n",
 "e": "import math\nimport numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: max_loops(src, 0, func=\"gradient_field\"))\n    imgs = np.stack([cv2.resize(i, (128, 128)).astype(np.float32) for i in (data.camera(), data.moon(), data.coins())])\n\n    def eq():\n        M, th, h = need(ns, \"gradient_field\")(torch.tensor(imgs), 9)\n        for i, im in enumerate(imgs):\n            gx = cv2.Sobel(im, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(im, cv2.CV_32F, 0, 1, ksize=3)\n            m = cv2.magnitude(gx, gy)\n            close(M[i].numpy(), m, atol=1e-2, what=f\"büyüklük {i}\")\n            t = np.mod(np.arctan2(gy, gx), np.pi)\n            b = np.minimum((t / np.pi * 9).astype(int), 8)\n            ref = np.bincount(b.ravel(), weights=m.ravel(), minlength=9)\n            close(h[i].numpy(), ref, rtol=1e-3, atol=1e-1, what=f\"histogram {i}\")\n        ensure(bool((th >= 0).all() and (th < math.pi).all()), \"θ ∈ [0, π)\")\n    R.test(\"büyüklük ve yönelim histogramları = cv2 Sobel ile\", eq)\n\n    def ramp():\n        yy, xx = torch.meshgrid(torch.arange(64.0), torch.arange(64.0), indexing=\"ij\")\n        a = math.radians(35)\n        img = (xx * math.cos(a) + yy * math.sin(a))[None]\n        M, th, h = need(ns, \"gradient_field\")(img, 18)\n        ensure(h[0].argmax().item() == 3, f\"35°'lik rampanın enerjisi 3. kutuda (30–40°) olmalı: {h[0].argmax().item()}\")\n    R.test(\"bilinen yönlü rampa doğru kutuda\", ramp)\n    if \"gradient_field\" not in ns:\n        return\n    X = torch.rand(64, 256, 256)\n    Xn = X.numpy()\n    t_loop = bench(lambda: [cv2.magnitude(cv2.Sobel(Xn[i], cv2.CV_32F, 1, 0), cv2.Sobel(Xn[i], cv2.CV_32F, 0, 1)) for i in range(64)], repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"gradient_field\"](X), repeat=3)\n    R.table(\"64 × 256 × 256\", [\"yöntem\", \"süre\"], [[\"cv2 döngüsü (yalnız büyüklük)\", fmt_time(t_loop)], [\"gradient_field (büyüklük + yön + histogram)\", fmt_time(t_vec)]])\n",
 "x1": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    ops = need(ns, \"operators\")()\n    f = data.camera().astype(np.float32)\n\n    def kernels():\n        close(cv2.filter2D(f, -1, ops[\"sobel\"][0]), cv2.Sobel(f, cv2.CV_32F, 1, 0, ksize=3), atol=1e-2, what=\"Sobel x\")\n        close(cv2.filter2D(f, -1, ops[\"sobel\"][1]), cv2.Sobel(f, cv2.CV_32F, 0, 1, ksize=3), atol=1e-2, what=\"Sobel y\")\n        close(cv2.filter2D(f, -1, ops[\"scharr\"][0]), cv2.Sobel(f, cv2.CV_32F, 1, 0, ksize=-1), atol=1e-2, what=\"Scharr x\")\n        ensure(ops[\"roberts\"][0].shape == (2, 2) and ops[\"prewitt\"][0].sum() == 0, \"Roberts 2×2, Prewitt sıfır toplamlı\")\n    R.test(\"çekirdekler OpenCV ile tutarlı\", kernels)\n\n    def orient():\n        angles = [5, 20, 35, 50, 65, 80]\n        rows = []\n        res = {}\n        for n in (\"prewitt\", \"sobel\", \"scharr\"):\n            e = need(ns, \"orientation_error\")(n, angles)\n            res[n] = float(np.max(e))\n            rows.append([n, f\"{np.mean(e):.2f}°\", f\"{np.max(e):.2f}°\"])\n        ensure(res[\"scharr\"] <= res[\"sobel\"] <= res[\"prewitt\"] + 1e-6, f\"Scharr ≤ Sobel ≤ Prewitt bekleniyordu: {res}\")\n        R.table(\"Sinüzoidal desende yön hatası (derece)\", [\"operatör\", \"ortalama\", \"en büyük\"], rows)\n    R.test(\"yön doğruluğu: Scharr en iyi\", orient)\n",
 "x2": "import numpy as np\nimport cv2\nfrom scipy import ndimage\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    img = data.camera()\n    f = img.astype(np.float64)\n    r = need(ns, \"pipeline\")(img)\n\n    def steps():\n        L = ndimage.correlate(f, np.array([[1, 1, 1], [1, -8, 1], [1, 1, 1]], float), mode=\"mirror\")\n        close(r[\"L\"], L, atol=1e-6, what=\"L\")\n        close(r[\"S1\"], f - L, atol=1e-6, what=\"S1\")\n        gx = ndimage.correlate(f, np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float), mode=\"mirror\")\n        gy = ndimage.correlate(f, np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], float), mode=\"mirror\")\n        G = np.abs(gx) + np.abs(gy)\n        close(r[\"G\"], G, atol=1e-6, what=\"G\")\n        Gs = ndimage.uniform_filter(G, 5, mode=\"mirror\")\n        close(r[\"Gs\"], Gs, atol=1e-6, what=\"Gs\")\n        close(r[\"P\"], -L * Gs / Gs.max(), atol=1e-6, what=\"P\")\n        close(r[\"S2\"], f - L * Gs / Gs.max(), atol=1e-6, what=\"S2\")\n        ensure(r[\"out\"].dtype == np.uint8, \"out uint8\")\n    R.test(\"tüm ara adımlar\", steps)\n\n    def noise():\n        flat = r[\"S1\"][ndimage.uniform_filter(f, 15) > 0] - f[ndimage.uniform_filter(f, 15) > 0]\n        n1 = np.abs(r[\"S1\"] - f)[ndimage.uniform_filter(r[\"G\"], 9) < np.percentile(r[\"G\"], 30)].mean()\n        n2 = np.abs(r[\"S2\"] - f)[ndimage.uniform_filter(r[\"G\"], 9) < np.percentile(r[\"G\"], 30)].mean()\n        ensure(n2 < 0.5 * n1, f\"düz bölgelerde ayrıntı eklemesi S1'de {n1:.2f}, S2'de {n2:.2f}: maske düz bölgeleri korumalı\")\n        return f\"düz bölgelerde ortalama değişim: Laplasyen keskinleştirme {n1:.2f}, maskeli {n2:.2f} gri seviye\"\n    R.test(\"maske gürültüyü düz bölgelerde bastırıyor\", noise)\n",
 "x3": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    def k3():\n        K = need(ns, \"cv_laplacian3\")()\n        f = data.camera().astype(np.float32)\n        close(cv2.filter2D(f, -1, np.asarray(K, np.float32)), cv2.Laplacian(f, cv2.CV_32F, ksize=3), atol=1e-2)\n        return \"cv2.Laplacian(ksize=3) çekirdeği:\\n\" + np.array2string(np.asarray(K))\n    R.test(\"deneyle bulunan çekirdek cv2.Laplacian(ksize=3) ile aynı sonucu veriyor\", k3)\n\n    def disp():\n        f = data.camera().astype(np.float32)\n        L = cv2.Laplacian(f, cv2.CV_32F, ksize=1)\n        full, mid = need(ns, \"laplacian_display\")(L)\n        ensure(full.dtype == np.uint8 and full.min() == 0 and full.max() == 255, \"tam aralık\")\n        ensure(mid.dtype == np.uint8 and abs(int(mid[np.abs(L) < 1e-6].mean()) - 128) <= 1, \"sıfır → 128\")\n    R.test(\"laplacian_display\", disp)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_4_sharpening as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Türevlerin ayrık hâli** (bir boyutta):

$$
\frac{\partial f}{\partial x}\approx f(x+1)-f(x),\qquad \frac{\partial^2 f}{\partial x^2}\approx f(x+1)+f(x-1)-2f(x).
$$

| yapı | birinci türev | ikinci türev |
|---|---|---|
| sabit bölge | 0 | 0 |
| rampa | sabit, sıfırdan farklı (kalın kenar) | rampa içinde 0, başında/sonunda zıt işaretli |
| basamak | tek tepe | çift tepe + **sıfır geçişi** (kenarın yeri) |
| ince çizgi, nokta | zayıf | çok güçlü |

**Laplasyen:** $\nabla^2f=f_{xx}+f_{yy}$; çekirdekleri $\begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix}$ (4-komşulu) ve köşegenleri de içeren $\begin{bmatrix}1&1&1\\1&-8&1\\1&1&1\end{bmatrix}$. **Keskinleştirme:** $g=f+c\,\nabla^2f$; merkez negatifse $c=-1$.

**unsharp masking:** bulanık kopyayı çıkararak maske $g_{mask}=f-\bar f$, sonra $g=f+k\,g_{mask}$. $k=1$ unsharp masking, $k>1$ highboost.

**Gradyan:** $\nabla f=(g_x,g_y)$, büyüklük $M=\sqrt{g_x^2+g_y^2}\approx|g_x|+|g_y|$, yön $\alpha=\operatorname{atan2}(g_y,g_x)$. Operatörler:

- Roberts (2×2, çapraz): $\begin{bmatrix}-1&0\\0&1\end{bmatrix}$, $\begin{bmatrix}0&-1\\1&0\end{bmatrix}$
- Prewitt: $[-1,0,1]$ türev × $[1,1,1]$ ortalama
- Sobel: $[-1,0,1]$ türev × $[1,2,1]$ yumuşatma

OpenCV'nin `Sobel` ve `Laplacian` fonksiyonları **korelasyon** uygular ve kenarda `BORDER_REFLECT_101` kullanır.

## 1 · Elle

<!-- çözüm -->

İnce çizgide ikinci türev $-12$: çizginin yüksekliğinin iki katı. Basamak kenarda (pozisyon 8–10) ikinci türev önce $+6$, sonra $-12$, sonra $+6$: kenarın her iki tarafını vurgulayan "çift kenar".

Sobel $g_x$ simetrik bir tepede 0'dır: soldaki ve sağdaki sütunların ağırlıklı toplamları eşit.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**1-boyutlu profil $f=(5,5,4,3,2,1,0,0,0,6,0,0)$; $f'(x)=f(x+1)-f(x)$, $f''(x)=f(x+1)+f(x-1)-2f(x)$**

- `d1ramp`: Rampa içinde $f'(3)$
- `d2ramp`: Rampa içinde $f''(3)$
- `d2start`: Rampa başında $f''(1)$
- `d2line`: İnce çizgide $f''(9)$

**Parça $\begin{bmatrix}10&10&10\\10&50&10\\10&10&10\end{bmatrix}$, merkez**

- `lap4`: 4-komşulu Laplasyen
- `sharp`: Keskinleştirilmiş $g=f-\nabla^2f$
- `sobx`: Sobel $g_x$ ($[-1,0,1]$ satırlar, $[1,2,1]$ ağırlık) bu parçada

**Unsharp masking: $f=100$, bulanık $\bar f=80$**

- `us`: $k=1$: $f+k(f-\bar f)$
- `hb`: Highboost $k=2.5$

In [ ]:
cevaplar = dict(
    d1ramp=None,
    d2ramp=None,
    d2start=None,
    d2line=None,
    lap4=None,
    sharp=None,
    sobx=None,
    us=None,
    hb=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"d1ramp":[-1,0],"d2ramp":[0,0],"d2start":[-1,0],"d2line":[-12,0],"lap4":[-160,0],"sharp":[210,0],"sobx":[0,0],"us":[120,0],"hb":[150,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python; görüntü liste listesi. Kenarda `BORDER_REFLECT_101` (cb|abc…, kenar tekrarlanmaz): indeks $-1\to1$, $W\to W-2$.

- `diff1(f)`, `diff2(f)` — 1-boyutlu liste; `diff1` uzunluğu $n-1$, `diff2` uzunluğu $n-2$ ($x=1..n-2$).
- `laplacian(img)` — 4-komşulu çekirdek.
- `sobel(img)` → `(gx, gy)`.

Kontrol `cv2.Laplacian(…, ksize=1)` ve `cv2.Sobel(…, ksize=3)` ile float32'de karşılaştırır.

In [ ]:
%%writefile stage_b.py
def diff1(f):
    ...


def diff2(f):
    ...


def laplacian(img):
    ...


def sobel(img):
    ...


f = [5, 5, 4, 3, 2, 1, 0, 0, 0, 6, 0, 0]
print(diff1(f), diff2(f))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy (bulanıklaştırma için `cv2.GaussianBlur` serbest, diğer filtreler NumPy ile), **döngü yok**.

- `sharpen_laplacian(img, kind=4)` — `kind` 4 ya da 8 (köşegenli); $g=f-\nabla^2f$, `uint8`'e kırp. Kenar `REFLECT_101`.
- `unsharp(img, k=1.0, ksize=5, sigma=0)` — `cv2.GaussianBlur` ile $\bar f$, $g=f+k(f-\bar f)$, `uint8`'e kırp (`np.rint`).
- `gradient_magnitude(img, norm="l2")` — Sobel ile $g_x,g_y$, `"l2"` ya da `"l1"`; float32.

In [ ]:
%%writefile stage_c.py
import numpy as np
import cv2


def sharpen_laplacian(img, kind=4):
    ...


def unsharp(img, k=1.0, ksize=5, sigma=0):
    ...


def gradient_magnitude(img, norm="l2"):
    ...


from skimage import data
img = data.moon()
print(sharpen_laplacian(img).dtype, unsharp(img, 2.0).mean(), gradient_magnitude(img).max())

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Türevlenebilir unsharp mask.

- `gaussian_kernel1d(sigma, ksize)` — `sigma` bir tensör olabilir, gradyan akar.
- `UnsharpMask(ksize=9)` — öğrenilebilir `log_sigma` ve `amount`; `forward(x)` ($(H,W)$, [0,1]) → $x+\text{amount}\,(x-G_\sigma*x)$, ayrılabilir `conv2d`, `reflect` doldurma, kırpma yok.
- `fit_unsharp(x, target, steps=400, lr=0.05)` → `(sigma, amount)`.

Kontrol bilinen $\sigma$ ve kazançla üretilmiş bir hedeften parametreleri geri bulmanı bekler.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn.functional as F


def gaussian_kernel1d(sigma, ksize):
    ...


class UnsharpMask(torch.nn.Module):
    def __init__(self, ksize=9):
        super().__init__()
        ...

    def forward(self, x):
        ...


def fit_unsharp(x, target, steps=400, lr=0.05):
    ...


from skimage import data
x = torch.tensor(data.moon(), dtype=torch.float32)[:128, :128] / 255
print(UnsharpMask()(x).shape)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`gradient_field(x, nbins=9)` — `x: (B, H, W)`; Sobel $g_x, g_y$ **tek** `conv2d` çağrısıyla (2 çıkış kanallı çekirdek), `reflect` doldurma; büyüklük $M$, işaretsiz yön $\theta\in[0,\pi)$; her görüntü için $M$ ağırlıklı, `nbins` kutulu yönelim histogramı (bir sonraki fazdaki HOG'un çekirdeği). Döner: `(M, theta, hist)`; `hist: (B, nbins)`. **Döngü yok** (`scatter_add_` ya da offset'li `bincount`).

In [ ]:
%%writefile stage_e.py
import math
import torch
import torch.nn.functional as F


def gradient_field(x, nbins=9):
    """x (B, H, W) → (M, theta, hist (B, nbins)). Döngü yok."""
    ...


x = torch.rand(2, 32, 32)
M, th, h = gradient_field(x)
print(M.shape, th.min().item() >= 0, h.shape)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Roberts, Prewitt, Sobel ve Scharr: çapraz kenarlar ve yön doğruluğu

- `operators()` → `{"roberts": (kx, ky), "prewitt": (kx, ky), "sobel": (kx, ky), "scharr": (kx, ky)}` NumPy çekirdekleri (korelasyon yönünde: $g_x$ soldan sağa artınca pozitif). Scharr: $[-3,0,3]$, $[-10,0,10]$, $[-3,0,3]$.
- `orientation_error(name, angles)` — her `angle` için o açıda bir doğrusal rampa görüntüsü üret, operatörle $g_x, g_y$'yi hesapla, merkezdeki $\operatorname{atan2}$ açısının gerçeğinden mutlak farkını (derece) döndür.

Sobel ve Scharr için `cv2.Sobel` (`ksize=3` ve `ksize=-1`) ile karşılaştırma yapılır; Scharr'ın yön hatasının en küçük olduğunu göreceksin.

In [ ]:
%%writefile extra_x1.py
import math
import numpy as np
import cv2


def operators():
    ...


def orientation_error(name, angles):
    ...


print({k: v[0].shape for k, v in operators().items()})
print(orientation_error("sobel", [10, 30, 60]))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Yöntemleri birleştirmek: Laplasyen + yumuşatılmış Sobel maskesi + gamma

Tek bir yöntem nadiren yeter. Gonzalez'in klasik örneğindeki sıra (kemik taraması):

1. $L$: Laplasyen (8-komşulu, köşegenli).
2. $S_1=f-L$: Laplasyenle keskinleştirilmiş (gürültülü).
3. $G$: Sobel gradyan büyüklüğü $|g_x|+|g_y|$.
4. $G_s$: $G$'nin 5 × 5 kutu ortalaması (gürültüsü azaltılmış kenar maskesi).
5. $P=S_1$'in ayrıntı kısmı $(-L)$ ile $G_s/\max G_s$'in çarpımı: ayrıntıyı yalnız kenarlarda tut.
6. $S_2=f+P$; 7. $\gamma=0.5$ ile güç dönüşümü (çıkış [0, 255]).

- `pipeline(img)` → sözlük: `L, S1, G, Gs, P, S2, out` (float64, son adım `uint8`). Kenarlar `REFLECT_101` (`cv2.filter2D`, `cv2.blur` serbest).

In [ ]:
%%writefile extra_x2.py
import numpy as np
import cv2


def pipeline(img):
    ...


from skimage import data
r = pipeline(data.camera())
print({k: (v.dtype, round(float(v.mean()), 1)) for k, v in r.items()})

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · OpenCV'nin `Laplacian(ksize=3)` çekirdeği ve Laplasyen görüntüsünü ölçekleme

- `cv_laplacian3()` — `cv2.Laplacian(img, CV_32F, ksize=3)` hangi 3 × 3 çekirdeği uygular? Çekirdeği **deneyle** bul: bir dürtü görüntüsüne uygula ve oku. (İpucu: köşegenli ama beklediğin gibi değil.)
- `laplacian_display(L)` — Laplasyen negatif değerler içerir; görüntülemek için en küçüğü 0'a, en büyüğü 255'e ölçekle (`uint8`, `np.rint`), ve ayrıca orta gri etrafında: $128+L\cdot127/\max|L|$ (`uint8`).

In [ ]:
%%writefile extra_x3.py
import numpy as np
import cv2


def cv_laplacian3():
    ...


def laplacian_display(L):
    ...


print(cv_laplacian3())

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Kenar korumalı keskinleştirme: Laplasyen piramidiyle yerel kontrast

[Paris, Hasinoff, Kautz — Local Laplacian Filters: Edge-aware Image Processing with a Laplacian Pyramid (SIGGRAPH 2011)](https://doi.org/10.1145/1964921.1964963)

Unsharp masking büyük kontrastlı kenarlarda hale (halo) üretir. Gauss/Laplasyen piramidini kur, ardından Local Laplacian Filter'ın basitleştirilmiş (az sayıda referans seviyeli) hızlı sürümünü uygula ve ayrıntı artırmayı halo olmadan yap.


**Adımlar**

1. `cv2.pyrDown`/`cv2.pyrUp` kullanmadan Gauss ve Laplasyen piramidi yaz; yeniden kurulumun orijinali verdiğini doğrula.
2. Uzaklığa bağlı bir yeniden eşleme fonksiyonu $r(i)$ ile her referans seviyesi için dönüştürülmüş görüntünün Laplasyen piramidini hesapla (hızlı yaklaşım: 8–16 seviye).
3. Her piramit katsayısını, Gauss piramidindeki yerel değere göre iki komşu referans seviyesi arasında interpolasyonla seç.
4. Unsharp masking ile aynı ayrıntı artışında halo genişliğini bir kenar profili üzerinde karşılaştır.

**Kabul ölçütleri**

- Piramit yeniden kurulum hatası $<10^{-5}$ (float).
- Basamak kenar profilinde aşma (overshoot) unsharp masking'in en fazla yarısı.
- 512 × 512 gri görüntüde süre < 2 s (NumPy, vektörel).

## Kaynaklar

- [Gonzalez & Woods — Digital Image Processing, Bölüm 3.6–3.7](https://www.imageprocessingplace.com/) — ©
- [OpenCV — Sobel Derivatives, Laplace Operator tutorial'ları](https://docs.opencv.org/4.x/d2/d2c/tutorial_sobel_derivatives.html) — Apache-2.0
- [Szeliski — Computer Vision: Algorithms and Applications, Bölüm 3.2.3](https://szeliski.org/Book/) — © (PDF ücretsiz)